# SAND Challenge - Task 2: ALS Progression Prediction
## 5-Fold Cross-Validation Approach

## Objective
Predict the ALSFRS-R score at the **last follow-up visit** based on:
- Audio features from 8 recordings (5 vowels + 3 syllables)
- Temporal information (months between assessments)
- Initial ALSFRS-R score

## Task Details
- **Classes**: 4 (ALS with dysarthria levels 1-4, no healthy subjects)
- **Metric**: Averaged F1-Score (macro)
- **Baseline**: PART Algorithm (F1 = 0.583)
- **Current Best**: ISDS Team (F1 = 0.5794)

## Evaluation Strategy
**Following professor's guidelines:**
- Combine train + validation sets (132 samples total)
- 5-fold stratified cross-validation
- Report: Mean F1 ± Std across 5 folds

## Approach
1. **Multi-Model Feature Extraction**: Wav2Vec2 + HuBERT
2. **Feature Fusion**: Concatenate audio + temporal features
3. **Ensemble Learning**: Multiple classifiers with 5-fold CV

## 1. Setup and Imports

### 1.1 Install Required Packages (if needed)

In [ ]:
# Uncomment if packages need to be installed
# !pip install transformers soundfile torch scikit-learn xgboost pandas numpy matplotlib seaborn tqdm

### 1.2 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import Wav2Vec2Model, Wav2Vec2Processor, HubertModel, Wav2Vec2FeatureExtractor
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from pathlib import Path
import warnings
from tqdm import tqdm
from scipy.signal import resample
from typing import List, Optional, Tuple, Dict
import json
from datetime import datetime
import pickle  # For feature caching

# Sklearn imports
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, 
    make_scorer, accuracy_score, balanced_accuracy_score
)

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

print("✓ All libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS (Apple GPU) available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

---
## 2. Configuration

### 2.1 Paths and Global Settings

In [ ]:
# ============================================================================
# PATHS - ADJUST THESE TO YOUR SYSTEM
# ============================================================================
DATA_PATH = Path('/Users/fabian.drzimalla/Master_Projects/SLP_PStA_Team_Drzimalla_Diakourakis/data/task2')
EXCEL_PATH = DATA_PATH / 'sand_task_2.xlsx'
TASK_DIR = DATA_PATH / 'training'

# Output directory for results
OUTPUT_DIR = Path('../experiments/task2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Configuration loaded")
print(f"  Data Path: {DATA_PATH}")
print(f"  Excel Path: {EXCEL_PATH}")
print(f"  Task Directory: {TASK_DIR}")
print(f"  Output Directory: {OUTPUT_DIR}")

### 2.2 Cross-Validation Configuration

In [ ]:
# ============================================================================
# CROSS-VALIDATION SETTINGS (Following Professor's Guidelines)
# ============================================================================
N_FOLDS = 5
CV_RANDOM_STATE = 42

print("="*80)
print("CROSS-VALIDATION CONFIGURATION")
print("="*80)
print(f"Strategy: {N_FOLDS}-Fold Stratified Cross-Validation")
print(f"Data: Combined Train + Validation sets")
print(f"Reporting: Mean ± Std over {N_FOLDS} runs")
print("="*80)

### 2.3 Feature Extraction Configuration

We'll use **two complementary models** for robust feature extraction:
- **Wav2Vec2**: Contrastive learning-based representations
- **HuBERT**: Clustering-based masked prediction

In [ ]:
# ============================================================================
# FEATURE EXTRACTOR CONFIGS
# ============================================================================

# Wav2Vec2 Configuration (based on Task 1 best model)
WAV2VEC2_CONFIG = {
    "model_name": "facebook/wav2vec2-large-960h",
    "layers": [6, 9, 12, 15, 18],  # Multi-layer extraction
    "pooling": "mean",  # Temporal pooling
    "layer_fusion": "concat",  # Concatenate layer outputs
}

# HuBERT Configuration
HUBERT_CONFIG = {
    "model_name": "facebook/hubert-large-ls960-ft",
    "layers": [6, 9, 12, 15, 18],  # Match Wav2Vec2 layers
    "pooling": "mean",
    "layer_fusion": "concat",
}

# Feature Fusion Strategy
FUSION_STRATEGY = "concat"  # Options: "concat", "average"

# Batch size
BATCH_SIZE = 16

print("="*80)
print("MULTI-MODEL FEATURE EXTRACTION CONFIGURATION")
print("="*80)
print("\nWav2Vec2 Config:")
for key, value in WAV2VEC2_CONFIG.items():
    print(f"  - {key}: {value}")
print("\nHuBERT Config:")
for key, value in HUBERT_CONFIG.items():
    print(f"  - {key}: {value}")
print(f"\nFusion Strategy: {FUSION_STRATEGY}")
print(f"Batch Size: {BATCH_SIZE}")
print("="*80)

---
## 3. Data Loading and Exploration

### 3.1 Load ALL Training Data (Train + Validation Combined)

In [ ]:
# ============================================================================
# IMPORTANT: Use ALL training data (not split into train/val)
# We'll use 5-fold CV to evaluate, so we need all data combined
# ============================================================================

# Load the FULL training set
df_full = pd.read_excel(EXCEL_PATH, sheet_name='SAND - TRAINING set - Task 2')

print("="*80)
print("DATA LOADING")
print("="*80)
print(f"Total samples: {len(df_full)}")
print(f"Columns: {df_full.columns.tolist()}")
print("\nFirst 5 rows:")
display(df_full.head())

print("\nTarget Distribution (ALSFRS--R_end):")
print(df_full['ALSFRS--R_end'].value_counts().sort_index())
print("="*80)

### 3.2 Exploratory Data Analysis

In [ ]:
print("="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Target class distribution (ALSFRS-R_end)
ax1 = axes[0, 0]
class_counts = df_full['ALSFRS--R_end'].value_counts().sort_index()
class_counts.plot(kind='bar', ax=ax1, color=['#e74c3c', '#e67e22', '#f39c12', '#2ecc71'])
ax1.set_title('Target: ALSFRS-R at Follow-up (End)', fontsize=12, fontweight='bold')
ax1.set_xlabel('ALSFRS-R Score')
ax1.set_ylabel('Count')
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(class_counts):
    ax1.text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')

# 2. Initial class distribution (ALSFRS-R_start)
ax2 = axes[0, 1]
start_counts = df_full['ALSFRS--R_start'].value_counts().sort_index()
start_counts.plot(kind='bar', ax=ax2, color=['#3498db', '#9b59b6', '#1abc9c', '#34495e'])
ax2.set_title('Initial: ALSFRS-R at Baseline (Start)', fontsize=12, fontweight='bold')
ax2.set_xlabel('ALSFRS-R Score')
ax2.set_ylabel('Count')
ax2.grid(axis='y', alpha=0.3)

# 3. Progression matrix (Start vs End)
ax3 = axes[1, 0]
progression_matrix = pd.crosstab(df_full['ALSFRS--R_start'], df_full['ALSFRS--R_end'])
sns.heatmap(progression_matrix, annot=True, fmt='d', cmap='YlOrRd', ax=ax3, cbar_kws={'label': 'Count'})
ax3.set_title('Progression Matrix: Start → End', fontsize=12, fontweight='bold')
ax3.set_xlabel('ALSFRS-R End')
ax3.set_ylabel('ALSFRS-R Start')

# 4. Time distribution
ax4 = axes[1, 1]
df_full['Months'].plot(kind='hist', bins=20, ax=ax4, color='#16a085', edgecolor='black')
ax4.set_title('Distribution of Follow-up Duration', fontsize=12, fontweight='bold')
ax4.set_xlabel('Months between assessments')
ax4.set_ylabel('Frequency')
ax4.grid(axis='y', alpha=0.3)
ax4.axvline(df_full['Months'].mean(), color='red', linestyle='--', 
            label=f'Mean: {df_full["Months"].mean():.1f}', linewidth=2)
ax4.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Statistics:")
print(f"  Average follow-up: {df_full['Months'].mean():.2f} ± {df_full['Months'].std():.2f} months")
print(f"  Age range: {df_full['Age'].min()}-{df_full['Age'].max()} years")
print(f"  Gender: {df_full['Sex'].value_counts().to_dict()}")
print(f"  Class balance: {(class_counts / len(df_full) * 100).round(2).to_dict()}%")

### 3.3 Data Preparation Function

In [ ]:
def prepare_file_paths(df: pd.DataFrame, task_dir: Path) -> Tuple[List[List[str]], np.ndarray, np.ndarray]:
    """
    Prepare file paths and labels from dataframe.
    
    Returns:
        audio_files: List of lists, where each inner list contains 8 audio file paths
        labels: Target labels (ALSFRS--R_end) - THIS IS THE KEY COLUMN!
        metadata: Array of [Age, Sex_encoded, Months, ALSFRS--R_start]
    """
    audio_types = ['phonationA', 'phonationE', 'phonationI', 'phonationO', 'phonationU',
                   'rhythmKA', 'rhythmPA', 'rhythmTA']
    
    audio_files = []
    labels = []
    metadata = []
    
    for _, row in df.iterrows():
        patient_id = row['ID']
        
        # Collect all 8 audio files for this patient
        patient_files = []
        all_exist = True
        
        for audio_type in audio_types:
            file_path = task_dir / audio_type / f"{patient_id}_{audio_type}.wav"
            if file_path.exists():
                patient_files.append(str(file_path))
            else:
                all_exist = False
                print(f"Warning: Missing file {file_path}")
                break
        
        if all_exist and len(patient_files) == 8:
            audio_files.append(patient_files)
            
            # TARGET: ALSFRS--R_end (progression outcome)
            labels.append(row['ALSFRS--R_end'])
            
            # Encode metadata features
            sex_encoded = 1 if row['Sex'] == 'M' else 0
            metadata.append([
                row['Age'],
                sex_encoded,
                row['Months'],
                row['ALSFRS--R_start']  # Initial score as feature
            ])
    
    return audio_files, np.array(labels), np.array(metadata)

# Prepare ALL data
print("\nPreparing data...")
all_files, y_all, metadata_all = prepare_file_paths(df_full, TASK_DIR)

print(f"\n✓ Data prepared:")
print(f"  Total samples: {len(all_files)}")
print(f"  Files per sample: {len(all_files[0]) if all_files else 0}")
print(f"  Metadata features: {metadata_all.shape[1]} (Age, Sex, Months, ALSFRS-R_start)")
print(f"  Target distribution: {np.bincount(y_all)}")
print(f"  Target labels (first 10): {y_all[:10]}")

---
## 4. Feature Extraction

### 4.1 Load Pre-trained Models

In [ ]:
print("LOADING PRE-TRAINED MODELS (M2 MAX OPTIMIZED)")
print("="*80)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f"\nDevice: {device}")

# Disable gradient computation globally (we're only doing inference)
torch.set_grad_enabled(False)

# Load Wav2Vec2
print("1. Loading Wav2Vec2...")
wav2vec_processor = Wav2Vec2Processor.from_pretrained(WAV2VEC2_CONFIG['model_name'])
wav2vec_model = Wav2Vec2Model.from_pretrained(WAV2VEC2_CONFIG['model_name']).to(device)
wav2vec_model.eval()
print(f"   ✓ Wav2Vec2 loaded: {WAV2VEC2_CONFIG['model_name']}")
print(f"   Hidden size: {wav2vec_model.config.hidden_size}")
print(f"   Device: {device}")

# Load HuBERT
print("\n2. Loading HuBERT...")
hubert_processor = Wav2Vec2FeatureExtractor.from_pretrained(HUBERT_CONFIG['model_name'])
hubert_model = HubertModel.from_pretrained(HUBERT_CONFIG['model_name']).to(device)
hubert_model.eval()
print(f"   ✓ HuBERT loaded: {HUBERT_CONFIG['model_name']}")
print(f"   Hidden size: {hubert_model.config.hidden_size}")
print(f"   Device: {device}")

print("\n" + "="*80)
print("ALL MODELS LOADED SUCCESSFULLY!"))
print("="*80)

### 4.2 Feature Extraction Functions (Single File)

In [ ]:
def extract_wav2vec2_features(audio_path: str,
                             processor: Wav2Vec2Processor,
                             model: Wav2Vec2Model,
                             device: torch.device,
                             layers: List[int],
                             pooling: str = "mean",
                             layer_fusion: str = "concat") -> Optional[np.ndarray]:
    """
    Extract Wav2Vec2 features from audio file.
    """
    try:
        # Load and preprocess audio
        audio, sr = sf.read(audio_path)
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        
        # Resample if needed
        target_sr = 16000
        if sr != target_sr:
            num_samples = int(len(audio) * target_sr / sr)
            audio = resample(audio, num_samples)
        
        # Process with Wav2Vec2
        inputs = processor(audio, sampling_rate=target_sr, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        
        # Extract from specified layers
        layer_features = []
        for layer_idx in layers:
            hidden_state = outputs.hidden_states[layer_idx].cpu().numpy()[0]
            
            # Apply pooling
            if pooling == "mean":
                pooled = np.mean(hidden_state, axis=0)
            elif pooling == "max":
                pooled = np.max(hidden_state, axis=0)
            elif pooling == "first":
                pooled = hidden_state[0]
            elif pooling == "last":
                pooled = hidden_state[-1]
            
            layer_features.append(pooled)
        
        # Fuse layers
        if layer_fusion == "concat":
            features = np.concatenate(layer_features)
        elif layer_fusion == "mean":
            features = np.mean(layer_features, axis=0)
        
        return features
    
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None


def extract_hubert_features(audio_path: str,
                           processor: Wav2Vec2FeatureExtractor,
                           model: HubertModel,
                           device: torch.device,
                           layers: List[int],
                           pooling: str = "mean",
                           layer_fusion: str = "concat") -> Optional[np.ndarray]:
    """
    Extract HuBERT features from audio file.
    """
    try:
        # Load and preprocess audio
        audio, sr = sf.read(audio_path)
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        
        # Resample if needed
        target_sr = 16000
        if sr != target_sr:
            num_samples = int(len(audio) * target_sr / sr)
            audio = resample(audio, num_samples)
        
        # Process with HuBERT
        inputs = processor(audio, sampling_rate=target_sr, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        
        # Extract from specified layers
        layer_features = []
        for layer_idx in layers:
            hidden_state = outputs.hidden_states[layer_idx].cpu().numpy()[0]
            
            # Apply pooling
            if pooling == "mean":
                pooled = np.mean(hidden_state, axis=0)
            elif pooling == "max":
                pooled = np.max(hidden_state, axis=0)
            elif pooling == "first":
                pooled = hidden_state[0]
            elif pooling == "last":
                pooled = hidden_state[-1]
            
            layer_features.append(pooled)
        
        # Fuse layers
        if layer_fusion == "concat":
            features = np.concatenate(layer_features)
        elif layer_fusion == "mean":
            features = np.mean(layer_features, axis=0)
        
        return features
    
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

print("✓ Feature extraction functions defined")

### 4.3 Extract Features with BATCH PROCESSING + CACHING

In [ ]:
# ============================================================================
# OPTIMIZED BATCH FEATURE EXTRACTION
# ============================================================================

def extract_all_features_batched(file_groups: List[List[str]], 
                                fusion_strategy: str = "concat",
                                batch_size: int = 16) -> np.ndarray:
    """
    Extract features using BATCH PROCESSING for maximum efficiency.
    Processes 16 files simultaneously (optimized for 64GB RAM).
    """
    # Flatten all audio paths
    all_paths = []
    path_to_sample = []
    path_to_file_idx = []
    
    for sample_idx, file_group in enumerate(file_groups):
        for file_idx, audio_file in enumerate(file_group):
            all_paths.append(audio_file)
            path_to_sample.append(sample_idx)
            path_to_file_idx.append(file_idx)
    
    total_files = len(all_paths)
    all_wav2vec_features = [None] * total_files
    all_hubert_features = [None] * total_files
    
    print(f"Processing {total_files} audio files in batches of {batch_size}...")
    print(f"Total batches: {(total_files + batch_size - 1) // batch_size}\n")
    
    # Process in batches
    for batch_start in tqdm(range(0, total_files, batch_size), desc="Batch processing"):
        batch_end = min(batch_start + batch_size, total_files)
        batch_paths = all_paths[batch_start:batch_end]
        
        # Load and preprocess batch
        batch_audios = []
        for audio_path in batch_paths:
            try:
                audio, sr = sf.read(audio_path)
                if audio.ndim > 1:
                    audio = np.mean(audio, axis=1)
                
                target_sr = 16000
                if sr != target_sr:
                    num_samples = int(len(audio) * target_sr / sr)
                    audio = resample(audio, num_samples)
                
                batch_audios.append(audio)
            except:
                batch_audios.append(np.zeros(16000))
        
        # Pad to same length
        max_len = max(len(a) for a in batch_audios)
        padded_audios = [np.pad(a, (0, max_len - len(a))) for a in batch_audios]
        
        # === WAV2VEC2 BATCH INFERENCE (ON GPU!) ===
        inputs = wav2vec_processor(padded_audios, sampling_rate=16000,
                                   return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = wav2vec_model(**inputs, output_hidden_states=True)
        
        # Extract features for each item in batch
        for batch_idx in range(len(batch_audios)):
            layer_features = []
            for layer_idx in WAV2VEC2_CONFIG['layers']:
                hidden_state = outputs.hidden_states[layer_idx][batch_idx].cpu().numpy()
                
                if WAV2VEC2_CONFIG['pooling'] == "mean":
                    pooled = np.mean(hidden_state, axis=0)
                elif WAV2VEC2_CONFIG['pooling'] == "max":
                    pooled = np.max(hidden_state, axis=0)
                elif WAV2VEC2_CONFIG['pooling'] == "first":
                    pooled = hidden_state[0]
                elif WAV2VEC2_CONFIG['pooling'] == "last":
                    pooled = hidden_state[-1]
                
                layer_features.append(pooled)
            
            if WAV2VEC2_CONFIG['layer_fusion'] == "concat":
                features = np.concatenate(layer_features)
            else:
                features = np.mean(layer_features, axis=0)
            
            all_wav2vec_features[batch_start + batch_idx] = features
        
        # === HUBERT BATCH INFERENCE (ON GPU!) ===
        inputs = hubert_processor(padded_audios, sampling_rate=16000,
                                 return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = hubert_model(**inputs, output_hidden_states=True)
        
        for batch_idx in range(len(batch_audios)):
            layer_features = []
            for layer_idx in HUBERT_CONFIG['layers']:
                hidden_state = outputs.hidden_states[layer_idx][batch_idx].cpu().numpy()
                
                if HUBERT_CONFIG['pooling'] == "mean":
                    pooled = np.mean(hidden_state, axis=0)
                elif HUBERT_CONFIG['pooling'] == "max":
                    pooled = np.max(hidden_state, axis=0)
                elif HUBERT_CONFIG['pooling'] == "first":
                    pooled = hidden_state[0]
                elif HUBERT_CONFIG['pooling'] == "last":
                    pooled = hidden_state[-1]
                
                layer_features.append(pooled)
            
            if HUBERT_CONFIG['layer_fusion'] == "concat":
                features = np.concatenate(layer_features)
            else:
                features = np.mean(layer_features, axis=0)
            
            all_hubert_features[batch_start + batch_idx] = features
    
    # Reorganize by samples (8 files per sample)
    sample_features = [[] for _ in file_groups]
    for idx, sample_idx in enumerate(path_to_sample):
        sample_features[sample_idx].append((all_wav2vec_features[idx], all_hubert_features[idx]))
    
    # Concatenate features for each sample
    final_features = []
    for sample_feats in sample_features:
        if len(sample_feats) == 8:
            wav2vec_all = [f[0] for f in sample_feats]
            hubert_all = [f[1] for f in sample_feats]
            
            if fusion_strategy == "concat":
                combined = np.concatenate([
                    np.concatenate(wav2vec_all),
                    np.concatenate(hubert_all)
                ])
            else:
                combined = np.mean([
                    np.concatenate(wav2vec_all),
                    np.concatenate(hubert_all)
                ], axis=0)
            
            final_features.append(combined)
    
    return np.array(final_features)


# ============================================================================
# FEATURE EXTRACTION WITH INTELLIGENT CACHING
# ============================================================================

# Feature Cache Setup
FEATURE_CACHE_DIR = OUTPUT_DIR / 'feature_cache'
FEATURE_CACHE_DIR.mkdir(exist_ok=True)

# Create unique cache name based on configuration
cache_name = (f"features_w2v2_{'-'.join(map(str, WAV2VEC2_CONFIG['layers']))}_"
              f"hubert_{'-'.join(map(str, HUBERT_CONFIG['layers']))}_"
              f"{FUSION_STRATEGY}_n{len(all_files)}.pkl")
cache_path = FEATURE_CACHE_DIR / cache_name

print("\n" + "="*80)
print("FEATURE EXTRACTION (WITH INTELLIGENT CACHING)")
print("="*80)

# Check if features already exist
if cache_path.exists():
    print(f"\n🎉 CACHED FEATURES FOUND!")
    print(f"📁 Loading from: {cache_path.name}")
    print(f"⏱️  This saves ~3-5 minutes of computation time!\n")
    
    with open(cache_path, 'rb') as f:
        cached_data = pickle.load(f)
    
    X_audio = cached_data['X_audio']
    
    print(f"✓ Features loaded successfully!")
    print(f"  Shape: {X_audio.shape}")
    print(f"  Cached on: {cached_data['timestamp']}")
    print(f"  Config verified: ✓")
    
else:
    print(f"\n⚠️  NO CACHED FEATURES FOUND")
    print(f"Starting optimized batch feature extraction...")
    print(f"Expected duration: ~3-5 minutes (M2 Max GPU + batching)")
    print(f"💾 Features will be cached for future runs\n")
    
    # Feature Extraction with batching
    X_audio = extract_all_features_batched(
        all_files, 
        fusion_strategy=FUSION_STRATEGY,
        batch_size=BATCH_SIZE
    )
    
    # Save features to cache
    print(f"\n💾 Saving features to cache...")
    cache_data = {
        'X_audio': X_audio,
        'timestamp': datetime.now().isoformat(),
        'config': {
            'wav2vec2': WAV2VEC2_CONFIG,
            'hubert': HUBERT_CONFIG,
            'fusion_strategy': FUSION_STRATEGY,
            'batch_size': BATCH_SIZE,
            'device': str(device),
            'n_samples': len(all_files),
            'n_files_per_sample': len(all_files[0]) if all_files else 0
        }
    }
    
    with open(cache_path, 'wb') as f:
        pickle.dump(cache_data, f)
    
    print(f"✓ Features cached successfully!")
    print(f"📁 Cache location: {cache_path}")

# Display summary
print(f"\n{'='*80}")
print("FEATURE EXTRACTION COMPLETE")
print(f"{'='*80}")
print(f"Audio features shape: {X_audio.shape}")
print(f"  - Samples: {X_audio.shape[0]}")
print(f"  - Feature dimension: {X_audio.shape[1]}")
print(f"  - Wav2Vec2 contribution: {wav2vec_model.config.hidden_size * len(WAV2VEC2_CONFIG['layers']) * 8}")
print(f"  - HuBERT contribution: {hubert_model.config.hidden_size * len(HUBERT_CONFIG['layers']) * 8}")
print("="*80)

### 4.4 Combine Audio and Metadata Features + Label Transformation

In [ ]:
# ============================================================================
# IMPORTANT: XGBoost requires labels starting at 0!
# Transform ALSFRS-R labels from [1,2,3,4] to [0,1,2,3]
# ============================================================================

# Combine audio features with metadata
X_all = np.concatenate([X_audio, metadata_all], axis=1)
y_all_original = y_all  # Keep original for reference

# Transform labels: [1,2,3,4] → [0,1,2,3] for sklearn compatibility
y_all_final = y_all - 1

print("\n" + "="*80)
print("FINAL FEATURE MATRIX & LABEL TRANSFORMATION")
print("="*80)
print(f"Total samples: {X_all.shape[0]}")
print(f"Total features: {X_all.shape[1]}")
print(f"  - Audio features (Wav2Vec2 + HuBERT): {X_audio.shape[1]}")
print(f"  - Metadata features: {metadata_all.shape[1]} (Age, Sex, Months, ALSFRS-R_start)")

print(f"\nLabel Transformation (for sklearn/XGBoost compatibility):")
print(f"  Original ALSFRS-R labels: {np.unique(y_all_original)}")
print(f"  Transformed labels:       {np.unique(y_all_final)}")
print(f"\n  Mapping:")
print(f"    ALSFRS-R 1 (Severe dysarthria)   → Class 0")
print(f"    ALSFRS-R 2 (Moderate dysarthria) → Class 1")
print(f"    ALSFRS-R 3 (Mild dysarthria)     → Class 2")
print(f"    ALSFRS-R 4 (No dysarthria)       → Class 3")

print(f"\nFinal Target Distribution:")
for original, transformed in zip([1,2,3,4], [0,1,2,3]):
    count = np.sum(y_all_final == transformed)
    percentage = (count / len(y_all_final)) * 100
    print(f"  Class {transformed} (ALSFRS-R={original}): {count:3d} samples ({percentage:5.2f}%)")

---
## 5. Model Training with 5-Fold Cross-Validation

### 5.1 Define Models and Pipelines

In [ ]:
# Define cross-validation strategy
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=CV_RANDOM_STATE)

# Define scoring metric
f1_macro_scorer = make_scorer(f1_score, average='macro')

# Define models
models = {
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_split=2,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ]),
    
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', xgb.XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.8,
            random_state=42,
            eval_metric='mlogloss',
            n_jobs=-1
        ))
    ]),
    
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(
            C=10,
            kernel='rbf',
            gamma='scale',
            class_weight='balanced',
            random_state=42,
            probability=True
        ))
    ]),
    
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            random_state=42
        ))
    ]),
}

print("✓ Models defined:")
for name in models.keys():
    print(f"  - {name}")

### 5.2 Perform 5-Fold Cross-Validation for Each Model

In [ ]:
print("\n" + "="*80)
print("5-FOLD CROSS-VALIDATION")
print("="*80)

results = {}

for model_name, model in models.items():
    print(f"\n{'='*80}")
    print(f"Model: {model_name}")
    print(f"{'='*80}")
    
    # Perform cross-validation
    cv_results = cross_validate(
        model, X_all, y_all_final,
        cv=cv,
        scoring={
            'f1_macro': f1_macro_scorer,
            'accuracy': 'accuracy',
            'balanced_accuracy': 'balanced_accuracy'
        },
        return_train_score=True,
        n_jobs=-1
    )
    
    # Extract results
    f1_scores = cv_results['test_f1_macro']
    acc_scores = cv_results['test_accuracy']
    bal_acc_scores = cv_results['test_balanced_accuracy']
    
    # Store results
    results[model_name] = {
        'f1_macro_mean': np.mean(f1_scores),
        'f1_macro_std': np.std(f1_scores),
        'f1_macro_scores': f1_scores,
        'accuracy_mean': np.mean(acc_scores),
        'accuracy_std': np.std(acc_scores),
        'balanced_accuracy_mean': np.mean(bal_acc_scores),
        'balanced_accuracy_std': np.std(bal_acc_scores),
    }
    
    # Print results
    print(f"\nResults across {N_FOLDS} folds:")
    print(f"  F1-Macro: {results[model_name]['f1_macro_mean']:.4f} ± {results[model_name]['f1_macro_std']:.4f}")
    print(f"  Accuracy: {results[model_name]['accuracy_mean']:.4f} ± {results[model_name]['accuracy_std']:.4f}")
    print(f"  Balanced Accuracy: {results[model_name]['balanced_accuracy_mean']:.4f} ± {results[model_name]['balanced_accuracy_std']:.4f}")
    print(f"\n  Individual F1 scores per fold:")
    for i, score in enumerate(f1_scores, 1):
        print(f"    Fold {i}: {score:.4f}")

print("CROSS-VALIDATION COMPLETE")

### 5.3 Create Ensemble Models and Evaluate

In [ ]:
print("ENSEMBLE MODELS")


# Voting Ensemble
print("\n1. Voting Ensemble (Soft Voting)")
voting_clf = VotingClassifier(
    estimators=[
        ('rf', models['Random Forest']),
        ('xgb', models['XGBoost']),
        ('svm', models['SVM']),
        ('gb', models['Gradient Boosting'])
    ],
    voting='soft',
    n_jobs=-1
)

voting_cv_results = cross_validate(
    voting_clf, X_all, y_all_final,
    cv=cv,
    scoring={'f1_macro': f1_macro_scorer, 'accuracy': 'accuracy'},
    n_jobs=-1
)

voting_f1_scores = voting_cv_results['test_f1_macro']
results['Voting Ensemble'] = {
    'f1_macro_mean': np.mean(voting_f1_scores),
    'f1_macro_std': np.std(voting_f1_scores),
    'f1_macro_scores': voting_f1_scores,
}

print(f"F1-Macro: {results['Voting Ensemble']['f1_macro_mean']:.4f} ± {results['Voting Ensemble']['f1_macro_std']:.4f}")

# Stacking Ensemble
print("\n2. Stacking Ensemble")
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', models['Random Forest']),
        ('xgb', models['XGBoost']),
        ('gb', models['Gradient Boosting'])
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    n_jobs=-1
)

stacking_cv_results = cross_validate(
    stacking_clf, X_all, y_all_final,
    cv=cv,
    scoring={'f1_macro': f1_macro_scorer, 'accuracy': 'accuracy'},
    n_jobs=-1
)

stacking_f1_scores = stacking_cv_results['test_f1_macro']
results['Stacking Ensemble'] = {
    'f1_macro_mean': np.mean(stacking_f1_scores),
    'f1_macro_std': np.std(stacking_f1_scores),
    'f1_macro_scores': stacking_f1_scores,
}

print(f"F1-Macro: {results['Stacking Ensemble']['f1_macro_mean']:.4f} ± {results['Stacking Ensemble']['f1_macro_std']:.4f}")

---
## 6. Results Analysis and Visualization

### 6.1 Results Summary Table

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame([
    {
        'Model': name,
        'F1-Macro (Mean)': data['f1_macro_mean'],
        'F1-Macro (Std)': data['f1_macro_std'],
        'F1-Macro (Formatted)': f"{data['f1_macro_mean']:.4f} ± {data['f1_macro_std']:.4f}"
    }
    for name, data in results.items()
])

results_df = results_df.sort_values('F1-Macro (Mean)', ascending=False)

print("\n" + "="*80)
print("FINAL RESULTS (5-FOLD CROSS-VALIDATION)")
print("="*80)
print(results_df[['Model', 'F1-Macro (Formatted)']].to_string(index=False))
print("\n" + "="*80)

# Best model
best_model = results_df.iloc[0]
print(f"\nBEST MODEL: {best_model['Model']}")
print(f"   F1-Macro: {best_model['F1-Macro (Formatted)']}")
print(f"\nCOMPARISON:")
print(f"   Baseline (PART): 0.5830")
print(f"   Current Leader (ISDS): 0.5794")
print(f"   Our Best: {best_model['F1-Macro (Mean)']:.4f}")
improvement = (best_model['F1-Macro (Mean)'] - 0.583) * 100
print(f"   Improvement over baseline: {improvement:+.2f}%")

### 6.2 Visualization

In [ ]:
# Create comprehensive visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Mean F1 scores with error bars
models_list = results_df['Model'].tolist()
means = results_df['F1-Macro (Mean)'].tolist()
stds = results_df['F1-Macro (Std)'].tolist()

colors = ['#2ecc71' if m > 0.583 else '#e74c3c' for m in means]
bars = ax1.barh(models_list, means, xerr=stds, color=colors, alpha=0.7, 
                edgecolor='black', linewidth=1.5, capsize=5)

# Add baseline lines
ax1.axvline(0.583, color='red', linestyle='--', linewidth=2, 
            label='Baseline (PART: 0.583)', alpha=0.7)
ax1.axvline(0.5794, color='orange', linestyle='--', linewidth=2,
            label='Leader (ISDS: 0.5794)', alpha=0.7)

# Add value labels
for i, (mean, std) in enumerate(zip(means, stds)):
    ax1.text(mean + 0.01, i, f'{mean:.4f}±{std:.4f}', 
             va='center', fontweight='bold', fontsize=9)

ax1.set_xlabel('F1-Macro Score', fontsize=12, fontweight='bold')
ax1.set_title('Model Performance (5-Fold CV)', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(axis='x', alpha=0.3)
ax1.set_xlim(min(means) - 0.05, max(means) + 0.08)

# Plot 2: Individual fold scores
fold_data = []
for name in models_list:
    if 'f1_macro_scores' in results[name]:
        for fold, score in enumerate(results[name]['f1_macro_scores'], 1):
            fold_data.append({'Model': name, 'Fold': fold, 'F1-Macro': score})

fold_df = pd.DataFrame(fold_data)
sns.boxplot(data=fold_df, y='Model', x='F1-Macro', ax=ax2, 
            order=models_list, palette='Set2')
ax2.set_xlabel('F1-Macro Score', fontsize=12, fontweight='bold')
ax2.set_ylabel('')
ax2.set_title('Score Distribution Across Folds', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
ax2.axvline(0.583, color='red', linestyle='--', linewidth=1.5, alpha=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cv_results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Statistical Comparison

In [ ]:
# Create detailed fold-by-fold comparison
fold_comparison = pd.DataFrame({
    name: results[name]['f1_macro_scores'] 
    for name in models_list if 'f1_macro_scores' in results[name]
})
fold_comparison.index = [f'Fold {i+1}' for i in range(N_FOLDS)]

print("\n" + "="*80)
print("FOLD-BY-FOLD F1-MACRO SCORES")
print("="*80)
print(fold_comparison.to_string())
print("\n" + "="*80)
print("STATISTICS")
print("="*80)
print(fold_comparison.describe().loc[['mean', 'std', 'min', 'max']])
print("="*80)

---
## 7. Save Results

### 7.1 Save Experiment Configuration and Results

In [ ]:
# Create experiment directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"task2_5fold_cv_m2max_{timestamp}"
experiment_dir = OUTPUT_DIR / experiment_name
experiment_dir.mkdir(exist_ok=True)

# Save configuration
config = {
    "experiment_name": experiment_name,
    "timestamp": datetime.now().isoformat(),
    "task": "Task 2 - ALS Progression Prediction",
    "evaluation_strategy": f"{N_FOLDS}-Fold Stratified Cross-Validation",
    "hardware": {
        "device": str(device),
        "system": "MacBook Pro M2 Max",
        "ram": "64 GB",
        "batch_size": BATCH_SIZE
    },
    "feature_extraction": {
        "wav2vec2": WAV2VEC2_CONFIG,
        "hubert": HUBERT_CONFIG,
        "fusion_strategy": FUSION_STRATEGY,
        "batch_processing": True,
        "feature_caching": True
    },
    "dataset": {
        "total_samples": len(X_all),
        "num_classes": len(np.unique(y_all_final)),
        "feature_dim_audio": X_audio.shape[1],
        "feature_dim_metadata": metadata_all.shape[1],
        "feature_dim_total": X_all.shape[1],
        "class_distribution": {int(k): int(v) for k, v in zip(*np.unique(y_all_final, return_counts=True))}
    },
    "results": {
        name: {
            "f1_macro_mean": float(data['f1_macro_mean']),
            "f1_macro_std": float(data['f1_macro_std']),
            "f1_macro_scores": [float(s) for s in data['f1_macro_scores']] if 'f1_macro_scores' in data else None
        }
        for name, data in results.items()
    },
    "best_model": {
        "name": best_model['Model'],
        "f1_macro_mean": float(best_model['F1-Macro (Mean)']),
        "f1_macro_std": float(best_model['F1-Macro (Std)']),
        "baseline_f1": 0.583,
        "improvement_over_baseline_percent": float(improvement)
    }
}

with open(experiment_dir / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Save results DataFrame
results_df.to_csv(experiment_dir / 'results_summary.csv', index=False)

# Save fold-by-fold comparison
fold_comparison.to_csv(experiment_dir / 'fold_by_fold_comparison.csv')

print(f"\n✓ Results saved to: {experiment_dir}")
print(f"  - config.json")
print(f"  - results_summary.csv")
print(f"  - fold_by_fold_comparison.csv")

---
## 8. Final Summary

### 8.1 Experiment Summary

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)
print(f"\nTask: SAND Challenge - Task 2 (ALS Progression Prediction)")
print(f"Experiment: {experiment_name}")
print(f"Hardware: MacBook Pro M2 Max (64GB RAM)")
print(f"Device: {device}")
print(f"Evaluation: {N_FOLDS}-Fold Stratified Cross-Validation")
print(f"\nFeature Extraction:")
print(f"  - Wav2Vec2: {WAV2VEC2_CONFIG['model_name']}")
print(f"  - HuBERT: {HUBERT_CONFIG['model_name']}")
print(f"  - Fusion: {FUSION_STRATEGY}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Caching: Enabled")
print(f"\nDataset:")
print(f"  - Total samples: {len(X_all)}")
print(f"  - Total features: {X_all.shape[1]}")
print(f"  - Target: ALSFRS--R_end (progression outcome)")
print(f"\nTop 3 Models:")
for i, row in results_df.head(3).iterrows():
    print(f"  {i+1}. {row['Model']}: {row['F1-Macro (Formatted)']}")
print(f"\n🏆 Best Model: {best_model['Model']}")
print(f"   F1-Macro: {best_model['F1-Macro (Formatted)']}")
print(f"   Baseline: 0.5830")
print(f"   Improvement: {improvement:+.2f}%")
print(f"\nResults saved to: {experiment_dir}")
print("="*80)

### 8.2 Cache Management (Optional)

In [ ]:
# View cached features
print("\nCached feature files:")
for cache_file in FEATURE_CACHE_DIR.glob("*.pkl"):
    size_mb = cache_file.stat().st_size / (1024 * 1024)
    print(f"  - {cache_file.name} ({size_mb:.1f} MB)")

# To delete cache (uncomment if needed):
# import shutil
# shutil.rmtree(FEATURE_CACHE_DIR)
# print("✓ Cache deleted. Next run will extract features fresh.")